# Notebook 01 — Exploração de Métricas de Serving Multimodal

## Objetivos
1. Gerar e carregar dataset sintético de métricas de inferência multimodal.
2. Realizar análise exploratória (EDA) das distribuições de latência por modalidade.
3. Calcular SLIs (Service Level Indicators) conforme Beyer et al. (2016).
4. Identificar padrões temporais e correlações entre métricas.

## Conceito-chave (Documento 04)
> "Os Service Level Indicators (SLIs) representam métricas quantitativas que capturam aspectos
> críticos do comportamento do sistema. Para sistemas de inferência de ML, os SLIs mais relevantes
> incluem latência de predição (tipicamente medida em percentis P50, P95 e P99), throughput
> (requisições por segundo), taxa de erros, e métricas específicas de ML como accuracy em
> produção e drift score."
> — Seção "Fundamentos de SRE para Machine Learning"

In [ ]:
# Imports
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_preprocessing import DataPreprocessor
from src.model import SLOMonitor

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)
%matplotlib inline

## 1. Geração e Carregamento do Dataset

O dataset simula 24 h de requisições em um pipeline multimodal (texto, imagem, áudio)
com duas versões de modelo: **v1-stable** (90 % do tráfego) e **v2-canary** (10 %),
conforme a arquitetura de referência (Figura 1) e a estratégia de canary deployment
descrita no material da aula.

In [ ]:
# Gerar dataset (idempotente)
dp = DataPreprocessor(seed=42)
data_path = os.path.join("..", "data", "raw", "dataset.csv")

if not os.path.exists(data_path):
    df = dp.generate_and_save(path=data_path, n_samples=10_000)
else:
    df = dp.load_data(data_path)

print(f"Shape: {df.shape}")
df.head()

## 2. Limpeza e Preparação de Features

In [ ]:
df = dp.clean_data(df)
df = dp.prepare_features(df)
print(f"Shape após limpeza: {df.shape}")
df.info()

## 3. Estatísticas Descritivas

Conforme discutido na seção *Monitoramento de Latência e Observabilidade em Produção*:
> "A latência de inferência deve ser medida em múltiplos percentis, pois a média pode
> mascarar degradações significativas na cauda da distribuição."

In [ ]:
# Estatísticas gerais de latência
latency_cols = ["preprocessing_ms", "inference_ms", "postprocessing_ms",
                "total_latency_ms", "queue_wait_ms"]
df[latency_cols].describe(percentiles=[0.5, 0.95, 0.99]).round(2)

## 4. Distribuição de Modalidades e Versões

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df["modality"].value_counts().plot.bar(ax=axes[0], color=["#3498db", "#e67e22", "#9b59b6"])
axes[0].set_title("Distribuição de Modalidades")
axes[0].set_ylabel("Contagem")

df["model_version"].value_counts().plot.bar(ax=axes[1], color=["steelblue", "coral"])
axes[1].set_title("Distribuição de Versões (Stable vs Canary)")
axes[1].set_ylabel("Contagem")

plt.tight_layout()
plt.show()

## 5. Distribuição de Latência por Modalidade

A arquitetura multimodal (Figura 1) utiliza workers especializados por modalidade,
cada um com perfil de latência distinto:
- **Text Worker** (LLM/NLP) — operações leves em CPU.
- **Image Worker** (CNN/ViT) — requer GPU; latência moderada.
- **Audio Worker** (ASR/TTS) — latência mais alta; benefício de GPU/TPU.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i, mod in enumerate(["text", "image", "audio"]):
    subset = df[df["modality"] == mod]["total_latency_ms"]
    axes[i].hist(subset, bins=60, color=["#3498db", "#e67e22", "#9b59b6"][i],
                 alpha=0.7, edgecolor="white")
    p99 = np.percentile(subset, 99)
    axes[i].axvline(p99, color="red", linestyle="--", label=f"P99={p99:.1f}ms")
    axes[i].axvline(200, color="black", linestyle=":", alpha=0.5, label="SLO=200ms")
    axes[i].set_title(f"Latência — {mod.capitalize()}")
    axes[i].set_xlabel("Latência (ms)")
    axes[i].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 6. Cálculo de SLIs

Implementa o cálculo de SLIs conforme Beyer et al. (2016) —
seção *Fundamentos de SRE para Machine Learning*.

In [ ]:
monitor = SLOMonitor(slo_latency_p99_ms=200.0, slo_availability=0.999)
slis = monitor.calculate_slis(df)

print("=== Service Level Indicators ===")
for k, v in slis.items():
    print(f"  {k:20s}: {v:.4f}")

## 7. Análise Temporal — Padrões de Carga

Visualizamos como a latência e utilização de CPU variam ao longo das 24 h,
essencial para calibrar regras de auto-scaling (seção *Azure ML Managed Endpoints*).

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

# Latência média por hora
hourly = df.groupby("hour").agg(
    latency_mean=("total_latency_ms", "mean"),
    latency_p99=("total_latency_ms", lambda x: np.percentile(x, 99)),
    cpu_mean=("cpu_utilization", "mean"),
    replicas_mean=("replica_count", "mean"),
    error_rate=("is_error", "mean"),
).reset_index()

axes[0].plot(hourly["hour"], hourly["latency_mean"], marker="o", label="Média")
axes[0].plot(hourly["hour"], hourly["latency_p99"], marker="s", color="red", label="P99")
axes[0].axhline(200, color="gray", linestyle="--", alpha=0.5)
axes[0].set_ylabel("Latência (ms)")
axes[0].set_title("Latência por Hora do Dia")
axes[0].legend()

axes[1].bar(hourly["hour"], hourly["cpu_mean"], color="orange", alpha=0.7)
axes[1].set_ylabel("CPU Utilization")
axes[1].set_title("Utilização de CPU por Hora")

axes[2].plot(hourly["hour"], hourly["replicas_mean"], marker="d", color="green")
axes[2].set_ylabel("Réplicas (média)")
axes[2].set_xlabel("Hora do dia")
axes[2].set_title("Réplicas Ativas por Hora")

plt.tight_layout()
plt.show()

## 8. Correlação entre Métricas

Verificar se métricas de infraestrutura (CPU, memória, fila) correlacionam-se
com latência — fundamental para definir sinais de auto-scaling adequados.

In [ ]:
corr_cols = ["preprocessing_ms", "inference_ms", "postprocessing_ms",
             "queue_wait_ms", "total_latency_ms",
             "cpu_utilization", "memory_utilization", "replica_count"]

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, ax=ax)
ax.set_title("Matriz de Correlação — Métricas de Serving")
plt.tight_layout()
plt.show()

## 9. Comparação Stable vs Canary (visão geral)

Primeira comparação visual entre as versões v1-stable e v2-canary.
No Notebook 02: aprofundamos com testes estatísticos formais.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for ver, color in [("v1-stable", "steelblue"), ("v2-canary", "coral")]:
    subset = df[df["model_version"] == ver]
    ax.hist(subset["total_latency_ms"], bins=60, alpha=0.6,
            label=f"{ver} (n={len(subset)})", color=color, edgecolor="white")

ax.axvline(200, color="red", linestyle="--", label="SLO P99 = 200 ms")
ax.set_xlabel("Latência Total (ms)")
ax.set_ylabel("Frequência")
ax.set_title("Distribuição de Latência — Stable vs Canary")
ax.legend()
plt.tight_layout()
plt.show()

## 10. Taxa de Erro por Modalidade e Versão

In [ ]:
error_table = (
    df.groupby(["modality", "model_version"])["is_error"]
    .agg(["sum", "count", "mean"])
    .rename(columns={"sum": "erros", "count": "total", "mean": "error_rate"})
    .round(4)
)
error_table

## Resumo

Neste notebook:
- Geramos e exploramos o dataset sintético de métricas multimodais.
- Calculamos SLIs (latência P50/P95/P99, disponibilidade, throughput).
- Identificamos que a modalidade **áudio** apresenta maior latência.
- Observamos padrão temporal de carga correlacionado com réplicas.
- A versão **canary (v2)** apresenta indícios de latência ligeiramente superior.

**Próximo:** [02_treinamento.ipynb](02_treinamento.ipynb) — Simulação de auto-scaling e canary deployment.